In [ ]:

import os
import requests

#  Load API key from environment variable
API_KEY = os.getenv("POKE_PRICE_TRACKER_API_KEY")

# Check if API key is loaded
if not API_KEY:
    print("Error: API key not found. Please set POKE_PRICE_TRACKER_API_KEY.")


In [ ]:
import os
import re
import time
import requests
from typing import Dict, Optional, Tuple

IOAPI_KEY = "ac4aadab-8d33-45ff-8ca2-666723786cfc"  

class FinalCardPricer:
    def __init__(self):
        self.pokemontcg_api_key = IOAPI_KEY
        self.api_calls_made = 0

    def parse_card_input(self, card_input: str) -> Tuple[str, Optional[str]]:
        """
        Parse card input to extract name and number.
        Examples:
        "Rayquaza V 194/203" -> ("Rayquaza V", "194")
        "Cynthia SV82" -> ("Cynthia", "SV82")
        """
        parts = card_input.strip().split()
        if not parts:
            return card_input, None

        last_part = parts[-1]

        if '/' in last_part and re.search(r'\d+', last_part):
            number = last_part.split('/')[0]
            name = ' '.join(parts[:-1])
            return name, number

        if re.match(r'^[A-Z]*\d+$', last_part):
            number = last_part
            name = ' '.join(parts[:-1]) if len(parts) > 1 else card_input
            return name, number

        return card_input, None

    def get_card_from_pokemontcg(self, name: str, number: Optional[str] = None, retries: int = 5) -> Optional[Dict]:
        """
        Get card details from pokemontcg.io with robust retry logic.
        """
        query_parts = [f'name:"{name}"']
        if number:
            query_parts.append(f'number:{number}')
        query = ' AND '.join(query_parts)

        url = "https://api.pokemontcg.io/v2/cards"
        headers = {"X-Api-Key": self.pokemontcg_api_key}

        for attempt in range(retries):
            try:
                print(f"Attempt {attempt + 1}/{retries} - Searching for: {name}" + (f" #{number}" if number else ""))
                
                # Gradually increase timeout with each retry
                timeout = min(10 + (attempt * 5), 30)
                
                response = requests.get(
                    url, 
                    headers=headers, 
                    params={"q": query, "pageSize": 5}, 
                    timeout=timeout
                )
                self.api_calls_made += 1
                
                # Handle different error codes
                if response.status_code in [504, 502, 503]:
                    wait_time = 2 ** attempt  # Exponential backoff: 2, 4, 8, 16 seconds
                    print(f"Server error {response.status_code}, waiting {wait_time}s before retry...")
                    time.sleep(wait_time)
                    continue
                elif response.status_code == 429:
                    wait_time = 60  # Rate limited, wait longer
                    print(f"Rate limited, waiting {wait_time}s...")
                    time.sleep(wait_time)
                    continue
                    
                response.raise_for_status()
                data = response.json().get("data", [])
                if not data:
                    print("No cards found matching the query")
                    return None
                    
                print(f"Found {len(data)} card(s)")
                return data[0]
                
            except requests.exceptions.Timeout as e:
                wait_time = 3 + (attempt * 2)  # Progressive wait: 3, 5, 7, 9, 11 seconds
                print(f"Timeout error on attempt {attempt + 1}: {e}")
                if attempt < retries - 1:
                    print(f"Waiting {wait_time}s before retry...")
                    time.sleep(wait_time)
                else:
                    print("All retry attempts failed due to timeout")
                    
            except requests.exceptions.ConnectionError as e:
                wait_time = 5 + (attempt * 3)  # Progressive wait for connection issues
                print(f"Connection error on attempt {attempt + 1}: {e}")
                if attempt < retries - 1:
                    print(f"Waiting {wait_time}s before retry...")
                    time.sleep(wait_time)
                else:
                    print("All retry attempts failed due to connection issues")
                    
            except requests.exceptions.RequestException as e:
                print(f"Request error on attempt {attempt + 1}: {e}")
                if attempt < retries - 1:
                    time.sleep(2)
                else:
                    print("All retry attempts failed")
                    
        return None

    def get_card_price(self, card_input: str) -> Optional[Dict]:
        """
        Get card details and available pricing from pokemontcg.io only.
        """
        print(f"Processing: {card_input}")
        name, number = self.parse_card_input(card_input)
        print(f"Parsed - Name: '{name}', Number: '{number}'")
        
        # Add delay between requests to be nice to the API
        if self.api_calls_made > 0:
            time.sleep(1)
            
        card_data = self.get_card_from_pokemontcg(name, number)
        if not card_data:
            print("Card not found on pokemontcg.io")
            return None

        # Extract pricing information if available
        tcgplayer_prices = card_data.get("tcgplayer", {}).get("prices", {})
        cardmarket_prices = card_data.get("cardmarket", {}).get("prices", {})
        
        # Try to get a market price from available sources
        price = None
        price_source = None
        
        # Check TCGPlayer prices (most common)
        if tcgplayer_prices:
            # Try different price types in order of preference
            for price_type in ["holofoil", "normal", "1stEditionHolofoil", "1stEditionNormal", "unlimitedHolofoil"]:
                if price_type in tcgplayer_prices:
                    market_price = tcgplayer_prices[price_type].get("market")
                    if market_price:
                        price = market_price
                        price_source = f"TCGPlayer ({price_type})"
                        break
        
        # Fallback to Cardmarket if no TCGPlayer price
        if not price and cardmarket_prices:
            market_price = cardmarket_prices.get("averageSellPrice") or cardmarket_prices.get("avg1") or cardmarket_prices.get("avg7") or cardmarket_prices.get("avg30")
            if market_price:
                price = market_price
                price_source = "Cardmarket"
        
        print(f"Found card: {card_data.get('name')} - Price: ${price if price else 'N/A'}")
        
        return {
            "card": card_data,
            "price": price,
            "price_source": price_source,
            "name": card_data.get("name"),
            "number": card_data.get("number"),
            "set": card_data.get("set", {}).get("name"),
            "rarity": card_data.get("rarity")
        }


# Simple interface
def get_card_price(card_input: str) -> Optional[float]:
    pricer = FinalCardPricer()
    result = pricer.get_card_price(card_input)
    if result and result.get("price") and result["price"] is not None:
        return float(result["price"])
    return None


# Example usage
if __name__ == "__main__":
    pricer = FinalCardPricer()
    test_cards = ["Rayquaza V 194/203", "Cynthia SV82"]
    for card in test_cards:
        price_data = pricer.get_card_price(card)
        if price_data:
            if price_data.get("price"):
                print(f"{card}: ${price_data['price']:.2f} (via {price_data.get('price_source', 'Unknown')})")
                print(f"  Set: {price_data.get('set')}, Rarity: {price_data.get('rarity')}")
            else:
                print(f"{card}: Card found but no price data available")
                print(f"  Set: {price_data.get('set')}, Rarity: {price_data.get('rarity')}")
        else:
            print(f"{card}: Not found")
        print()

Processing: Rayquaza V 194/203
Parsed - Name: 'Rayquaza V', Number: '194'
Attempt 1/5 - Searching for: Rayquaza V #194
Request error on attempt 1: 404 Client Error: Not Found for url: https://api.pokemontcg.io/v2/cards?q=name%3A%22Rayquaza+V%22+AND+number%3A194&pageSize=5
Attempt 2/5 - Searching for: Rayquaza V #194
Found 1 card(s)
Found card: Rayquaza V - Price: $285.75
Rayquaza V 194/203: $285.75 (via TCGPlayer (holofoil))
  Set: Evolving Skies, Rarity: Rare Ultra

Processing: Cynthia SV82
Parsed - Name: 'Cynthia', Number: 'SV82'
Attempt 1/5 - Searching for: Cynthia #SV82
Timeout error on attempt 1: HTTPSConnectionPool(host='api.pokemontcg.io', port=443): Read timed out. (read timeout=10)
Waiting 3s before retry...
Attempt 2/5 - Searching for: Cynthia #SV82


In [ ]:
#******************************************************************************************